# Candidate Data Validation

This notebook validates synthetic recruiting candidates.

The candidate table contains:

- Application sources
- Education levels
- Previous experience
- Candidate locations
- Candidate profiles reserved for future accepted applications

The main rules are:

- Candidate IDs are complete, unique, and sequential.
- The table contains 40,000 candidates.
- Application sources use approved categories.
- Education levels agree with the employee data.
- Years of experience remain between 0 and 30.
- Candidate locations use a City, State format.
- The first 10,000 candidates are aligned with the employee population for future hiring records.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"


candidates = pd.read_csv(
    RAW_DATA_DIR / "candidates.csv"
)

employees = pd.read_csv(
    RAW_DATA_DIR / "employees.csv"
)

locations = pd.read_csv(
    RAW_DATA_DIR / "locations.csv"
)


print(
    "Candidates:",
    candidates.shape,
)

print(
    "Employees:",
    employees.shape,
)

print(
    "Locations:",
    locations.shape,
)

Candidates: (40000, 5)
Employees: (10000, 15)
Locations: (5, 5)


## 1. Initial inspection

In [2]:
candidates.head(10)

,candidate_id,application_source,education_level,years_experience,candidate_location
0,200001,Company Careers Page,Bachelor's,12,"Austin, Texas"
1,200002,Employee Referral,Master's,12,"Austin, Texas"
2,200003,Indeed,High School,10,"Reno, Nevada"
3,200004,LinkedIn,Master's,10,"Austin, Texas"
4,200005,Company Careers Page,Master's,8,"Austin, Texas"
5,200006,Indeed,Master's,21,"Buffalo, New York"
6,200007,Company Careers Page,Doctorate,8,"Austin, Texas"
7,200008,LinkedIn,Associate,21,"Austin, Texas"
8,200009,LinkedIn,Master's,18,"Reno, Nevada"
9,200010,Employee Referral,Associate,19,"Pittsburgh, PA"


In [3]:
candidates.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   candidate_id        40000 non-null  int64
 1   application_source  40000 non-null  str  
 2   education_level     40000 non-null  str  
 3   years_experience    40000 non-null  int64
 4   candidate_location  40000 non-null  str  
dtypes: int64(2), str(3)
memory usage: 1.5 MB


## 2. Structure and primary-key checks

In [4]:
expected_columns = [
    "candidate_id",
    "application_source",
    "education_level",
    "years_experience",
    "candidate_location",
]

expected_candidate_ids = list(
    range(
        200_001,
        240_001,
    )
)

actual_candidate_ids = (
    candidates
    .sort_values("candidate_id")[
        "candidate_id"
    ]
    .astype(int)
    .tolist()
)

structure_checks = pd.Series(
    {
        "table has 40,000 rows": (
            len(candidates)
            == 40_000
        ),
        "table has five columns": (
            len(candidates.columns)
            == 5
        ),
        "columns are in the expected order": (
            candidates.columns.tolist()
            == expected_columns
        ),
        "candidate IDs are complete": (
            candidates[
                "candidate_id"
            ].notna().all()
        ),
        "candidate IDs are unique": (
            candidates[
                "candidate_id"
            ].is_unique
        ),
        "candidate IDs are sequential": (
            actual_candidate_ids
            == expected_candidate_ids
        ),
        "table has no missing values": (
            not candidates
            .isna()
            .any()
            .any()
        ),
    },
    name="passed",
)

structure_checks

table has 40,000 rows                True
table has five columns               True
columns are in the expected order    True
candidate IDs are complete           True
candidate IDs are unique             True
candidate IDs are sequential         True
table has no missing values          True
Name: passed, dtype: bool

## 3. Application-source checks

In [5]:
allowed_application_sources = {
    "LinkedIn",
    "Employee Referral",
    "Company Careers Page",
    "Indeed",
    "University Recruiting",
    "Staffing Agency",
    "Professional Association",
    "Job Fair",
}

source_checks = pd.Series(
    {
        "application sources are valid": (
            set(
                candidates[
                    "application_source"
                ]
            )
            .issubset(
                allowed_application_sources
            )
        ),
        "all source categories appear": (
            allowed_application_sources
            .issubset(
                set(
                    candidates[
                        "application_source"
                    ]
                )
            )
        ),
    },
    name="passed",
)

source_checks

application sources are valid    True
all source categories appear     True
Name: passed, dtype: bool

In [6]:
application_source_summary = (
    candidates[
        "application_source"
    ]
    .value_counts()
    .rename_axis(
        "application_source"
    )
    .reset_index(
        name="candidate_count"
    )
)

application_source_summary[
    "percentage"
] = (
    application_source_summary[
        "candidate_count"
    ]
    / len(candidates)
    * 100
).round(2)

application_source_summary

,application_source,candidate_count,percentage
0,LinkedIn,11093,27.73
1,Company Careers Page,8320,20.80
2,Indeed,6698,16.74
3,Employee Referral,5263,13.16
4,University Recruiting,2884,7.21
5,Staffing Agency,2590,6.48
6,Professional Association,1716,4.29
7,Job Fair,1436,3.59


## 4. Education-level checks

In [7]:
valid_education_levels = set(
    employees[
        "education_level"
    ]
    .astype(str)
)

education_checks = pd.Series(
    {
        "candidate education levels are valid": (
            set(
                candidates[
                    "education_level"
                ].astype(str)
            )
            .issubset(
                valid_education_levels
            )
        )
    },
    name="passed",
)

education_checks

candidate education levels are valid    True
Name: passed, dtype: bool

In [8]:
education_summary = (
    candidates[
        "education_level"
    ]
    .value_counts()
    .rename_axis(
        "education_level"
    )
    .reset_index(
        name="candidate_count"
    )
)

education_summary

,education_level,candidate_count
0,Bachelor's,19081
1,Master's,8442
2,Associate,5906
3,High School,5675
4,Doctorate,896


## 5. Experience checks

In [9]:
experience_checks = pd.Series(
    {
        "experience is not missing": (
            candidates[
                "years_experience"
            ].notna().all()
        ),
        "experience is at least zero": (
            candidates[
                "years_experience"
            ].ge(0).all()
        ),
        "experience is at most 30": (
            candidates[
                "years_experience"
            ].le(30).all()
        ),
        "experience uses whole numbers": (
            candidates[
                "years_experience"
            ]
            .astype(float)
            .mod(1)
            .eq(0)
            .all()
        ),
    },
    name="passed",
)

experience_checks

experience is not missing        True
experience is at least zero      True
experience is at most 30         True
experience uses whole numbers    True
Name: passed, dtype: bool

In [10]:
candidates[
    "years_experience"
].describe()

count    40000.000000
mean         9.344825
std          5.808250
min          0.000000
25%          4.000000
50%          9.000000
75%         14.000000
max         30.000000
Name: years_experience, dtype: float64

In [11]:
experience_by_education = (
    candidates
    .groupby(
        "education_level"
    )
    .agg(
        candidate_count=(
            "candidate_id",
            "count",
        ),
        average_experience=(
            "years_experience",
            "mean",
        ),
        minimum_experience=(
            "years_experience",
            "min",
        ),
        maximum_experience=(
            "years_experience",
            "max",
        ),
    )
    .round(2)
)

experience_by_education

,candidate_count,average_experience,minimum_experience,maximum_experience
education_level,,,,
Associate,5906,8.39,0,24
Bachelor's,19081,9.44,0,30
Doctorate,896,12.29,0,25
High School,5675,7.51,0,24
Master's,8442,10.72,0,23


## 6. Candidate-location checks

In [12]:
company_locations = set(
    (
        locations[
            "city"
        ].astype(str).str.strip()
        + ", "
        + locations[
            "state"
        ].astype(str).str.strip()
    )
)

location_checks = pd.Series(
    {
        "candidate locations are complete": (
            candidates[
                "candidate_location"
            ].notna().all()
        ),
        "candidate locations are nonempty": (
            candidates[
                "candidate_location"
            ]
            .astype(str)
            .str.strip()
            .ne("")
            .all()
        ),
        "locations contain a comma": (
            candidates[
                "candidate_location"
            ]
            .str.contains(
                r",\s*\S+",
                regex=True,
            )
            .all()
        ),
        "company locations appear among candidates": (
            company_locations
            .issubset(
                set(
                    candidates[
                        "candidate_location"
                    ]
                )
            )
        ),
    },
    name="passed",
)

location_checks

candidate locations are complete             True
candidate locations are nonempty             True
locations contain a comma                    True
company locations appear among candidates    True
Name: passed, dtype: bool

In [13]:
candidate_location_summary = (
    candidates[
        "candidate_location"
    ]
    .value_counts()
    .head(15)
    .rename_axis(
        "candidate_location"
    )
    .reset_index(
        name="candidate_count"
    )
)

candidate_location_summary

,candidate_location,candidate_count
0,"Austin, Texas",5610
1,"Fremont, California",5314
2,"Phoenix, Arizona",4839
3,"Reno, Nevada",4543
4,"Buffalo, New York",4215
5,"San Francisco, CA",702
6,"Seattle, WA",701
7,"Nashville, TN",686
8,"Atlanta, GA",672
9,"Miami, FL",670


## 7. Future-hire candidate alignment

The first 10,000 candidates are reserved for applications that will later be accepted and connected to the 10,000 employees.

Candidate and employee IDs remain different because the two IDs identify different stages of the person's relationship with the company.

In [14]:
reserved_candidates = (
    candidates
    .sort_values(
        "candidate_id"
    )
    .head(
        len(employees)
    )
    .reset_index(
        drop=True
    )
)

employee_order = (
    employees
    .sort_values(
        "employee_id"
    )
    .reset_index(
        drop=True
    )
)

reserved_candidate_checks = pd.Series(
    {
        "there are 10,000 reserved candidates": (
            len(
                reserved_candidates
            )
            == 10_000
        ),
        "reserved candidate education matches employees": (
            reserved_candidates[
                "education_level"
            ]
            .astype(str)
            .eq(
                employee_order[
                    "education_level"
                ].astype(str)
            )
            .all()
        ),
    },
    name="passed",
)

reserved_candidate_checks

there are 10,000 reserved candidates              True
reserved candidate education matches employees    True
Name: passed, dtype: bool

In [15]:
future_hire_mapping_preview = pd.DataFrame(
    {
        "candidate_id": (
            reserved_candidates[
                "candidate_id"
            ].head(10)
        ),
        "employee_id": (
            employee_order[
                "employee_id"
            ].head(10)
        ),
        "candidate_education": (
            reserved_candidates[
                "education_level"
            ].head(10)
        ),
        "employee_education": (
            employee_order[
                "education_level"
            ].head(10)
        ),
    }
)

future_hire_mapping_preview

,candidate_id,employee_id,candidate_education,employee_education
0,200001,100001,Bachelor's,Bachelor's
1,200002,100002,Master's,Master's
2,200003,100003,High School,High School
3,200004,100004,Master's,Master's
4,200005,100005,Master's,Master's
5,200006,100006,Master's,Master's
6,200007,100007,Doctorate,Doctorate
7,200008,100008,Associate,Associate
8,200009,100009,Master's,Master's
9,200010,100010,Associate,Associate


## 8. Complete validation summary

In [16]:
all_checks = pd.concat(
    [
        structure_checks,
        source_checks,
        education_checks,
        experience_checks,
        location_checks,
        reserved_candidate_checks,
    ]
)

validation_results = pd.DataFrame(
    {
        "check": all_checks.index,
        "passed": all_checks.values,
    }
)

validation_results

,check,passed
0,"table has 40,000 rows",True
1,table has five columns,True
2,columns are in the expected order,True
3,candidate IDs are complete,True
4,candidate IDs are unique,True
5,candidate IDs are sequential,True
6,table has no missing values,True
7,application sources are valid,True
8,all source categories appear,True
9,candidate education levels are valid,True


In [17]:
if validation_results[
    "passed"
].all():
    print(
        "All candidate validation "
        "checks passed."
    )
else:
    print(
        "One or more candidate "
        "validation checks failed."
    )

All candidate validation checks passed.


## 9. Conclusions

The synthetic candidate table successfully represents the first stage of the recruiting system.

### Successful checks

- The table contains 40,000 candidate profiles.
- Candidate IDs are complete, unique, and sequential.
- Candidate IDs range from 200001 through 240000.
- Application sources use approved categories.
- Candidate education levels agree with the workforce education categories.
- Years of experience remain between 0 and 30.
- Candidate locations use a City, State format.
- The first 10,000 candidates align with the employee population for future accepted applications.
- All candidate validation checks passed.

### Current simplifications

- Candidate names and contact information are intentionally excluded.
- Candidate demographic characteristics are excluded.
- Candidate profiles do not yet identify the jobs for which they applied.
- Hire outcomes are not stored in the candidate table.
- The first 10,000 candidate records are reserved for later employee mappings.
- Applications and job requisitions will be generated in later checkpoints.